# 8. k-Means Clustering
**Use case:** Financial Services — Retail banking customer segmentation for personalized products

A regional retail bank sends the same credit card and savings offers to every customer, and response rates are low while some customers receive offers that do not fit their situation. The bank holds balances, monthly deposits, card spend, digital login share, loan-to-balance ratio, number of products and overdraft history for each customer. Marketing wants a few clear, stable segments with names they can design offers for, and compliance wants the segments to avoid proxying for protected classes. Success means well-separated, stable segments, each with a sensible offer and a measurable expected margin.

> 💡 Think of a youth soccer coach with 40 kids and 4 practice cones. Each kid runs to the nearest cone, then each cone is moved to the middle of its crowd, and the kids re-pick their nearest cone. After a few rounds nobody switches and you have four natural groups.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Run k-Means by hand on 12 bank customers
Implement the assignment and update steps in plain numpy and watch 12 customers settle into 3 segments.

| 5W + H | |
|---|---|
| **Who** | A marketing analyst learning what clustering actually does. |
| **What** | Twelve typed-in customers with average balance and monthly card transactions. |
| **When** | Before the first segmentation workshop with the marketing team. |
| **Where** | A short numpy script; no libraries beyond numpy. |
| **Why** | To see that k-Means is just 'join the nearest centre, move the centre' repeated until stable. |
| **How** | Standardize two columns, start from a deliberately poor set of centres, loop over assign and update steps printing labels and inertia each round. |

**🧠 ML concepts:** Distance-based grouping, Why features must be scaled, Assignment and update steps, Inertia as a quality measure, Convergence  
**🐍 Python concepts:** numpy arrays and axis arguments, Broadcasting with None to add an axis, argmin to find the nearest centre, List comprehension inside np.array, break out of a for loop, Filtering a list with zip

In [ ]:
import numpy as np

# 🐍 PY: each row is one bank customer: [avg_monthly_balance_k, card_txns_per_month]
customers = np.array([
    [1.2, 45], [0.8, 50], [1.5, 40], [1.0, 55],      # young, low balance, card-heavy
    [25.0, 8], [30.0, 5], [22.0, 10], [28.0, 6],     # older savers, high balance, few txns
    [8.0, 25], [10.0, 22], [9.0, 28], [7.5, 20],     # mid-life families
])
# 🐍 PY: a list of strings, one per customer, so we can label rows when printing
names = [f"C{i:02d}" for i in range(len(customers))]

# 🧠 ML: scale columns so 'balance in $k' and 'txn count' contribute fairly to distance
# 🐍 PY: numpy broadcasting subtracts each column's mean from every row at once
X = (customers - customers.mean(axis=0)) / customers.std(axis=0)

k = 3
# 🧠 ML: start with k centroids picked from the data (rows 0, 1, 4 - a deliberately poor start)
centroids = X[[0, 1, 4]].copy()

# 🐍 PY: range(5) loops five times; k-Means usually settles in a handful of rounds
for it in range(5):
    # 🧠 ML: distance from every customer to every centroid -> shape (12, 3)
    # 🐍 PY: X[:, None, :] adds an axis so broadcasting compares all pairs in one line
    dist = np.sqrt(((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
    # 🧠 ML: assignment step - each customer joins its nearest centroid
    labels = dist.argmin(axis=1)
    # 🧠 ML: update step - each centroid moves to the average of its members
    new_c = np.array([X[labels == j].mean(axis=0) for j in range(k)])
    # 🧠 ML: inertia = total squared distance to own centroid; smaller = tighter segments
    inertia = ((X - new_c[labels]) ** 2).sum()
    print(f"iter {it}: labels={labels.tolist()} inertia={inertia:.3f}")
    # 🐍 PY: np.allclose checks if two arrays are (almost) equal -> stop when nothing moves
    if np.allclose(new_c, centroids):
        print("Converged - centroids stopped moving.")
        break
    centroids = new_c

print("\nSegment profiles (original units):")
for j in range(k):
    members = customers[labels == j]
    # 🐍 PY: list comprehension with a filter picks the names in this segment
    who = [n for n, l in zip(names, labels) if l == j]
    print(f"Segment {j}: avg balance ${members[:, 0].mean():5.1f}k, "
          f"{members[:, 1].mean():4.1f} card txns/mo -> {who}")

**Try this next:**
- Remove the scaling line (use X = customers.astype(float)) and see how balance alone drives the result.
- Set k = 2 and k = 4 and compare final inertia.
- Start from rows 0, 4, 8 and count how many iterations convergence takes.

## MEDIUM: Segment 2,500 customers and pick k with elbow and silhouette
Use scikit-learn KMeans on realistic synthetic banking data, choose k with the elbow and silhouette scores, and map segments to offers.

| 5W + H | |
|---|---|
| **Who** | The bank's customer analytics team. |
| **What** | 2,500 synthetic customers with age, balance, card activity, digital share, loan ratio and number of products. |
| **When** | During campaign planning for the next quarter. |
| **Where** | A notebook against a monthly customer snapshot from the data warehouse. |
| **Why** | Marketing needs a small number of meaningful segments to design offers for. |
| **How** | StandardScaler, KMeans for k=2 to 7, print inertia and silhouette, pick the best k, profile segments with groupby and assign offers with simple rules. |

**🧠 ML concepts:** Feature scaling with StandardScaler, Elbow method, Silhouette score, Multiple restarts with n_init, Profiling clusters to give them business meaning  
**🐍 Python concepts:** List of dicts as configuration, Looping and appending DataFrames then pd.concat, np.clip for valid ranges, max with key=dict.get, groupby().mean() and size(), iterrows for row-wise rules

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)

# 🐍 PY: a list of dicts describes each hidden persona; we generate customers from them
personas = [
    dict(n=900, age=27, bal=2.5, card=45, digital=0.9, loan=0.1, prod=1.5),   # digital young
    dict(n=700, age=42, bal=12.0, card=30, digital=0.6, loan=0.7, prod=3.5),  # mortgage families
    dict(n=500, age=63, bal=55.0, card=10, digital=0.3, loan=0.1, prod=2.5),  # affluent savers
    dict(n=400, age=48, bal=0.8, card=12, digital=0.25, loan=1.0, prod=1.2),   # stretched borrowers
]
frames = []
for p in personas:
    n = p["n"]
    # 🐍 PY: np.clip keeps generated numbers inside sensible business ranges
    frames.append(pd.DataFrame({
        "age": np.clip(np.random.normal(p["age"], 6, n), 18, 90).round(),
        "avg_balance_k": np.abs(np.random.normal(p["bal"], p["bal"] * 0.35, n)).round(2),
        "card_txns_month": np.random.poisson(p["card"], n),
        "digital_login_share": np.clip(np.random.normal(p["digital"], 0.12, n), 0, 1).round(2),
        "loan_balance_ratio": np.clip(np.random.normal(p["loan"], 0.15, n), 0, 1.5).round(2),
        "num_products": np.clip(np.random.poisson(p["prod"], n), 1, 8),
    }))
df = pd.concat(frames, ignore_index=True)
print(f"Customers: {len(df)}  features: {list(df.columns)}")

# 🧠 ML: k-Means uses distance, so every feature must be on the same scale (mean 0, std 1)
scaler = StandardScaler()
X = scaler.fit_transform(df)

# 🧠 ML: elbow = inertia drops fast then flattens; silhouette = how well-separated (-1..1, higher better)
print("\n k   inertia  silhouette")
results = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    # 🧠 ML: silhouette on a 1500-row sample is accurate enough and much faster
    sil = silhouette_score(X, km.labels_, sample_size=1500, random_state=42)
    results[k] = sil
    print(f"{k:2d} {km.inertia_:9.0f}  {sil:8.3f}  {'*' * int(sil * 40)}")

# 🐍 PY: max with key= returns the dict key whose value is largest
best_k = max(results, key=results.get)
print(f"\nBest k by silhouette: {best_k}")

km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X)
df["segment"] = km.labels_

# 🐍 PY: groupby().mean() builds one profile row per segment; size() counts members
profile = df.groupby("segment").mean().round(2)
profile["customers"] = df.groupby("segment").size()
print("\nSegment profiles:")
print(profile.to_string())

# 🧠 ML: clusters have no names - a human reads the profile and assigns a business meaning
print("\nSuggested offers:")
for seg, row in profile.iterrows():
    if row.avg_balance_k > 30:
        offer = "Wealth advisory / CDs"
    elif row.loan_balance_ratio > 0.4 and row.avg_balance_k < 5:
        offer = "Debt consolidation + budgeting tools"
    elif row.loan_balance_ratio > 0.4:
        offer = "Mortgage refinance + 529 college saver"
    else:
        offer = "Cash-back card + mobile-first savings"
    print(f"  Segment {seg} ({int(row.customers)} cust, age {row.age:.0f}): {offer}")

**Try this next:**
- Remove StandardScaler and see how the best k and segments change.
- Drop the age column from the model and check whether the segments stay similar.
- Merge the stretched borrowers into the mortgage persona (same loan ratio) and see if silhouette still picks 4.

## HARD: Stable, explainable and fair segmentation pipeline
Build a production segmentation pipeline with log transforms, elbow plus silhouette selection, bootstrap stability, a surrogate tree, a baseline comparison and an expected-margin view.

| 5W + H | |
|---|---|
| **Who** | The customer analytics lead, marketing and the fair-lending compliance reviewer. |
| **What** | 5,000 synthetic customers with long-tailed money columns and behaviour features; age is kept out of the model for a fairness review. |
| **When** | Before segments are written to the CRM and used for credit and deposit offers. |
| **Where** | The bank's analytics platform; the saved pipeline scores every customer monthly. |
| **Why** | Segments must be well separated, stable when refreshed, explainable to marketing and not a proxy for protected classes. |
| **How** | Pipeline of log1p plus StandardScaler plus KMeans(k-means++), elbow and silhouette table, bootstrap ARI stability, balance-tier baseline, z-score fingerprints, surrogate decision tree, expected margin per offer and joblib save. |

**🧠 ML concepts:** Log transform for skewed money features, k selection with elbow and silhouette, Cluster stability via bootstrap and Adjusted Rand Index, Comparing against a simple baseline segmentation, Surrogate decision tree for explainability, Fairness review with held-out protected attributes  
**🐍 Python concepts:** Tuple unpacking in a for loop over persona specs, Functions returning DataFrames, Pipeline slicing with pipe[:-1], pct_change for elbow drops, Dict of tuples for offer economics, tempfile and joblib for model saving

In [ ]:
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore")
SEED = 42
SKEWED = ["avg_balance_k", "monthly_deposits_k", "card_spend_k"]
OTHER = ["digital_login_share", "loan_balance_ratio", "num_products", "overdrafts_12m"]
FEATS = SKEWED + OTHER


def make_customers(seed=SEED):
    rng = np.random.default_rng(seed)
    # (n, balance, deposits, card spend, digital, loan ratio, products, overdraft rate, mean age)
    # 🧠 ML: four hidden personas - a good segmentation should rediscover them without being told
    personas = [(1800, 2.5, 3.0, 1.8, 0.9, 0.1, 1.5, 0.5, 27), (1400, 12, 7.5, 2.5, 0.6, 0.7, 3.5, 0.3, 42),
                (1000, 55, 12, 3.5, 0.3, 0.1, 2.5, 0.05, 63), (800, 0.8, 2.2, 0.6, 0.25, 1.0, 1.2, 4.0, 48)]
    frames = []
    for n, bal, dep, spend, dig, loan, prod, od, age in personas:
        frames.append(pd.DataFrame({
            # 🧠 ML: lognormal gives long right tails, like real bank balances
            "avg_balance_k": rng.lognormal(np.log(bal), 0.5, n),
            "monthly_deposits_k": rng.lognormal(np.log(dep), 0.35, n),
            "card_spend_k": rng.lognormal(np.log(spend), 0.4, n),
            "digital_login_share": np.clip(rng.normal(dig, 0.12, n), 0, 1),
            "loan_balance_ratio": np.clip(rng.normal(loan, 0.15, n), 0, 1.5),
            "num_products": np.clip(rng.poisson(prod, n), 1, 8),
            "overdrafts_12m": rng.poisson(od, n),
            "age": np.clip(rng.normal(age, 7, n), 18, 90).round(),  # NOT a model input (fairness)
        }))
    return pd.concat(frames, ignore_index=True)


def build_pipeline(k, seed=SEED):
    # 🧠 ML: log1p squashes long-tailed money columns so a few millionaires do not own a cluster
    pre = ColumnTransformer([("log", FunctionTransformer(np.log1p), SKEWED),
                             ("keep", "passthrough", OTHER)])
    # 🧠 ML: k-means++ init + n_init restarts avoid the bad-start trap seen in the easy level
    return Pipeline([("pre", pre), ("scale", StandardScaler()),
                     ("km", KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=seed))])


# 🐍 PY: range(2, 9) is a default argument; it yields k = 2..8
def choose_k(df, ks=range(2, 9)):
    rows = []
    for k in ks:
        pipe = build_pipeline(k).fit(df[FEATS])
        # 🐍 PY: pipe[:-1] slices the pipeline to everything except KMeans - the scaled features
        Z = pipe[:-1].transform(df[FEATS])
        # 🧠 ML: silhouette on a 2,000-row sample keeps it fast; it is O(n squared) on full data
        sil = silhouette_score(Z, pipe["km"].labels_, sample_size=2000, random_state=SEED)
        rows.append((k, pipe["km"].inertia_, sil))
    out = pd.DataFrame(rows, columns=["k", "inertia", "silhouette"])
    # 🧠 ML: elbow = where the % drop in inertia from adding one more cluster becomes small
    out["inertia_drop_%"] = (-out.inertia.pct_change() * 100).round(1)
    return out


def stability(df, k, seeds=(1, 2, 3, 4)):
    # 🧠 ML: refit on 80% bootstraps; ARI near 1 = segments are real, not a random artefact
    ref = build_pipeline(k).fit(df[FEATS])
    scores = []
    for s in seeds:
        sub = df.sample(frac=0.8, random_state=s)
        alt = build_pipeline(k, seed=s).fit(sub[FEATS])
        scores.append(adjusted_rand_score(ref.predict(sub[FEATS]), alt.predict(sub[FEATS])))
    return np.mean(scores)


df = make_customers()
print(f"Customers={len(df)}  model features={len(FEATS)} (age held out for fairness review)")
table = choose_k(df)
print(table.round(3).to_string(index=False))
# 🐍 PY: idxmax returns the row label of the highest silhouette; .loc then reads its k
best_k = int(table.loc[table.silhouette.idxmax(), "k"])
print(f"Chosen k={best_k}  bootstrap stability ARI={stability(df, best_k):.3f}")

# 🧠 ML: refit on all customers with the chosen k; predict assigns each to its nearest centroid
model = build_pipeline(best_k).fit(df[FEATS])
df["segment"] = model.predict(df[FEATS])

# 🧠 ML: baseline = the classic 'balance tier' segmentation many banks start with
# 🐍 PY: pd.qcut cuts balances into equal-sized quantile buckets (the old-school tiers)
tiers = pd.qcut(df.avg_balance_k, best_k, labels=False)
Zall = model[:-1].transform(df[FEATS])
print(f"Silhouette  k-means={silhouette_score(Zall, df.segment, sample_size=2000, random_state=SEED):.3f}"
      f"  vs balance-tier baseline={silhouette_score(Zall, tiers, sample_size=2000, random_state=SEED):.3f}")

# 🧠 ML: explain segments by z-score of each feature vs the whole book (+ = above average)
Z = pd.DataFrame(model[:-1].transform(df[FEATS]), columns=FEATS)
Z["segment"] = df.segment
print("\nSegment fingerprint (z-scores):")
print(Z.groupby("segment").mean().round(2).to_string())

# 🧠 ML: surrogate tree turns clusters into readable rules for marketing and model-risk review
# 🧠 ML: fidelity = how often the readable tree agrees with k-Means; above 0.95 is a trustworthy summary
sur = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(df[FEATS], df.segment)
print(f"\nSurrogate tree fidelity={sur.score(df[FEATS], df.segment):.3f}; first rules:")
print("\n".join(export_text(sur, feature_names=FEATS, decimals=1).splitlines()[:8]))

# 🧠 ML: business value - expected campaign margin per segment (response rate x margin x size)
offers = {"wealth": (0.06, 900), "refi": (0.04, 1200), "card": (0.08, 150), "hardship": (0.10, 60)}


def pick_offer(r):
    if r.avg_balance_k > 30:
        return "wealth"
    if r.overdrafts_12m > 2:
        return "hardship"
    return "refi" if r.loan_balance_ratio > 0.4 else "card"


# 🧠 ML: medians resist outliers better than means when profiling skewed money columns
prof = df.groupby("segment")[FEATS + ["age"]].median()
prof["size"] = df.segment.value_counts().sort_index()
print("\nseg  size  med_age  offer     expected_margin")
for seg, r in prof.iterrows():
    o = pick_offer(r)
    rate, margin = offers[o]
    print(f"{seg:3d} {int(r['size']):5d} {r.age:7.0f}  {o:9s} ${r['size'] * rate * margin:>10,.0f}")

# Fair lending: segments must not proxy for protected classes (ECOA/Reg B, UDAAP). Age is reviewed
# AFTER clustering; a segment that is mostly one age band needs a compliance sign-off before credit offers.
print("\nFairness check: age is held out; review segments that skew by age before credit offers.")
# 🧠 ML: save the full pipeline so next month's customers get the same transforms and centroids
path = os.path.join(tempfile.mkdtemp(), "retail_segments_kmeans.joblib")
joblib.dump(model, path)
print("Saved ->", os.path.basename(path))

**Try this next:**
- Remove the log1p step and compare silhouette and stability.
- Force k=3 and see which two personas merge and how expected margin changes.
- Compute the share of customers over 60 in each segment and flag any segment above 70 percent for compliance review.